In [1]:
import pandas as pd
from collections import Counter
import numpy as np
from pathlib import Path
from scipy.stats import entropy  # entropy is a function to compute KL-divergence

In [39]:
#Define QID - question matching -- WVS wave 7
qids = ['Q36','Q125','Q106','Q28','Q111','Q121','Q108','Q130','Q198','Q235']
name_dict = {
    "Q36":  "Gay Parenting",
    "Q125": "Refugee Asylum",
    "Q106": "Income Equality",
    "Q28":  "Maternal Employment",
    "Q111": "Environment vs Economy",
    "Q121": "Immigration Impact",
    "Q108": "Government Responsibility",
    "Q130": "Immigration Policy",
    "Q198": "Government Surveillance",
    "Q235": "Strong Leader",
}
# qids = ['Q36','Q125','Q106']
def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id +".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["Q36", "Q121"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        elif id in ["Q125"]:
            answers = {i: value_counts[i] for i in range(0, 3)}
        elif id in ["Q111"]:
            answers = {i: value_counts[i] for i in range(1, 4)}
        elif id in ["Q106","Q108"]:
            answers = {i: value_counts[i] for i in range(1, 11)}
        else:
            answers = {i: value_counts[i] for i in range(1, 5)} #4

        value_list.append({name_dict[id]: answers})

    return value_list


In [ ]:
country = 'NLD'
# country = 'DEU'
country = 'GBR'
# GPT-4.1
model_name = 'gpt4.1'
# T=0
replicate_path = f"../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_wvs_results/{country}/full_results_wvs7_original_{country}_"
reformulated_path = f"../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_wvs_reform_results/{country}/full_results_wvs7_reformulated_{country}_"

 


In [62]:
def get_human_counts(path, qids): #getting results from wvs 7
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["Q36", "Q121"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          elif id in ["Q125"]:
               answers = {i: human_counts[i] for i in range(0, 3)}
          elif id in ["Q111"]:
               answers = {i: human_counts[i] for i in range(1, 4)}
          elif id in ["Q106","Q108"]:
               answers = {i: human_counts[i] for i in range(1, 11)}
          else:
               answers = {i: human_counts[i] for i in range(1, 5)} #4
          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts(f"../data/WVS_Wave_7_{country}.csv", qids)

In [ ]:
replicate_results = get_counts(replicate_path, qids)
reformulated_results = get_counts(reformulated_path, qids)

In [68]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
#Get all the results and divergences
jsd_results_replicate = jsd_divergence_between_sets(human_results, replicate_results, base=2)
jsd_results_reformulated = jsd_divergence_between_sets(human_results, reformulated_results, base=2)


diff_reformulated_vs_replicate = {key: (jsd_results_reformulated[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}


In [ ]:
# format data
# Difference in JS-divergence between replicate and 4 experimental conditions
results_mapping = [
    ('Replicate', jsd_results_replicate),
    ('Reformulated - Replicate', diff_reformulated_vs_replicate),
    # ('Reformulated', jsd_results_reformulated),
]

# question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']
question_order = name_dict.values()
question_order = [
    'Environment vs Economy',    # ← Climate Change
    # Current Economy — no WVS counterpart (skipped)
    'Refugee Asylum',            # ← Refugee Allowing
    'Government Responsibility', # ← Health Insurance
    'Income Equality',           # ← Income Inequality
    'Immigration Impact',        # ← Race Diversity
    # Drug Addiction — no WVS counterpart (skipped)
    'Gay Parenting',             # ← Gay Marriage
    # Gun Regulation — no WVS counterpart (skipped)
    'Maternal Employment',       # ← Gender Role
    # WVS-only topics (no ANES analog)
    'Immigration Policy',
    'Government Surveillance',
    'Strong Leader',
]
sig_df = pd.DataFrame({'Question': question_order})

for column_name, result_dict in results_mapping:
    # Extract values from the dictionary in the precise 'question_order'
    ordered_values = [
        result_dict.get(q, np.nan)  # Use .get() and np.nan for missing keys
        for q in question_order
    ]
    # Add the column to the DataFrame, rounding to 3 decimals
    sig_df[column_name] = np.round(ordered_values, 3)

In [73]:
print(country)
sig_df

GBR


,Question,Replicate,Reformulated - Replicate
0,Environment vs Economy,0.049,-0.020
1,Refugee Asylum,0.232,-0.010
2,Government Responsibility,0.126,-0.031
3,Income Equality,0.193,-0.034
4,Immigration Impact,0.110,-0.027
5,Gay Parenting,0.122,-0.013
6,Maternal Employment,0.127,-0.010
7,Immigration Policy,0.101,-0.017
8,Government Surveillance,0.363,-0.104
9,Strong Leader,0.279,-0.148


In [74]:
def jsd_divergence_for_bootstrapping(human_sample, bootstrap_sample, base=2):
    p_counts = human_sample
    q_counts = bootstrap_sample

    p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
    q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

    p /= p.sum()
    q /= q.sum()

    epsilon = 1e-10
    p = np.clip(p, epsilon, 1)
    q = np.clip(q, epsilon, 1)

    m = 0.5 * (p + q)
    jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base))

    return jsd_pq


### CIs (Human vs conditions)

In [75]:
human_df = pd.read_csv(f"../data/WVS_Wave_7_{country}.csv")

def bootstrap_jsd(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    responses = np.array(question_df['Response'].dropna())

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True) #sample with replacement
        
        counts = Counter(sample)

        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base) #compute JSD for this iteration
        jsd_samples.append(jsd_val) #append to the samples list

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples  # <— raw samples for delta CI

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])
    
    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))

 

In [ ]:
#Create CIs (human vs conditions) and compare them
ci_replicate = []
ci_reformulated = []

for qid in qids:
    ci_replicate.append((name_dict[qid], bootstrap_jsd(path = replicate_path, qid = qid)))
    ci_reformulated.append((name_dict[qid], bootstrap_jsd(path = reformulated_path, qid = qid)))

ci_replicate = [{item[0] : item[1]} for item in ci_replicate]
ci_reformulated = [{item[0] : item[1]} for item in ci_reformulated]

ci_replicate = {list(d.keys())[0]: list(d.values())[0] for d in ci_replicate}
ci_reformulated = {list(d.keys())[0]: list(d.values())[0] for d in ci_reformulated}

In [ ]:
# Mean JS divergence and 95% bootstrap confidence intervals between ANSE and silicon distributions by item and condition.
def format_ci_data(ci_tuple):
    # Convert the list of tuples structure into a dictionary for easy access
    if ci_tuple is None:
        return "N/A"
    data = dict(ci_tuple)
    
    mean = data['mean']
    ci_lower = data['ci_lower']
    ci_upper = data['ci_upper']

    mean_str = f"{mean:.4f}"
    lower_str = f"{ci_lower:4f}"
    upper_str = f"{ci_upper:.4f}"

    return f"{mean_str} [{lower_str}, {upper_str}]"

results_mapping = [
    ('Replicate', ci_replicate),
    ('Reformulated', ci_reformulated),
]
ci_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_formatted_values = [
        format_ci_data(result_dict.get(q,None))
        for q in question_order
    ]

    ci_df[column_name] = ordered_formatted_values


In [79]:
ci_df

,Question,Replicate,Reformulated
0,Environment vs Economy,"0.0493 [0.042938, 0.0563]","0.0294 [0.024409, 0.0350]"
1,Refugee Asylum,"0.2323 [0.232320, 0.2323]","0.2229 [0.218386, 0.2275]"
2,Government Responsibility,"0.1265 [0.117117, 0.1366]","0.0956 [0.086854, 0.1054]"
3,Income Equality,"0.1941 [0.182622, 0.2053]","0.1597 [0.148603, 0.1711]"
4,Immigration Impact,"0.1103 [0.100461, 0.1206]","0.0838 [0.074989, 0.0931]"
5,Gay Parenting,"0.1219 [0.111639, 0.1326]","0.1094 [0.100468, 0.1186]"
6,Maternal Employment,"0.1269 [0.122447, 0.1309]","0.1167 [0.112973, 0.1209]"
7,Immigration Policy,"0.1017 [0.096256, 0.1075]","0.0843 [0.080264, 0.0888]"
8,Government Surveillance,"0.3635 [0.348502, 0.3781]","0.2593 [0.245253, 0.2738]"
9,Strong Leader,"0.2793 [0.272347, 0.2867]","0.1317 [0.121443, 0.1421]"
